# Week 2: Data Preprocessing and Feature Engineering

## Project Title
AI-Based Bike Rental Demand Prediction Using Machine Learning

## Objective
The objective of this week is to preprocess the Capital Bikeshare trip-history dataset and prepare it for machine learning-based bike rental demand prediction.

The October 2025 dataset contains individual bike rental records. These records will be cleaned, analyzed, and aggregated into hourly rental demand.

## Dataset
- Source: Capital Bikeshare System Data
- Dataset: October 2025 trip-history data
- File: `202510-capitalbikeshare-tripdata.csv`
- Original records: 624,869
- Original features: 13

## Tools and Libraries
- Python
- Google Colab
- Pandas and NumPy
- Matplotlib and Seaborn
- Scikit-learn

## Main Activities
1. Dataset loading and inspection
2. Data cleaning and missing-value handling
3. Outlier detection
4. Exploratory data analysis
5. Hourly demand aggregation
6. Feature extraction and engineering
7. Categorical encoding and numerical scaling
8. Exporting the processed dataset

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")

## 1. Dataset Loading

The Capital Bikeshare October 2025 dataset is loaded into a Pandas DataFrame. The dataset contains individual bike rental records, including ride timestamps, bike types, station details, geographical coordinates, and customer categories.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# Load the uploaded dataset

file_name = "202510-capitalbikeshare-tripdata.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

## 2. Dataset Understanding

The dataset is examined to understand its structure, column names, data types, and statistical information. This helps identify the data that requires cleaning and transformation.


In [ ]:
df.head()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
missing_values = df.isnull().sum()

print("Missing Values:")
print(missing_values)

## 3. Data Cleaning

Data cleaning is performed to improve the quality and consistency of the dataset.

The following operations are carried out:

- Remove duplicate ride records.
- Convert timestamp columns into datetime format.
- Check for invalid timestamps.
- Calculate ride duration.
- Identify missing values.
- Investigate unusually long rides.

Rows with missing station details are retained because their ride timestamps are still useful for hourly demand prediction.


In [ ]:
# Keep the original DataFrame unchanged

cleaned_df = df.copy()

print("Working copy created.")

In [ ]:
print("Duplicate ride IDs before cleaning:",
      cleaned_df['ride_id'].duplicated().sum())

cleaned_df = cleaned_df.drop_duplicates(
    subset='ride_id'
).copy()

print("Shape after duplicate removal:", cleaned_df.shape)

In [ ]:
cleaned_df['started_at'] = pd.to_datetime(
    cleaned_df['started_at'],
    errors='coerce'
)

cleaned_df['ended_at'] = pd.to_datetime(
    cleaned_df['ended_at'],
    errors='coerce'
)

print(cleaned_df[['started_at', 'ended_at']].dtypes)

In [ ]:
before = len(cleaned_df)

cleaned_df = cleaned_df.dropna(
    subset=['started_at', 'ended_at']
).copy()

print("Invalid timestamp records removed:",
      before - len(cleaned_df))

In [ ]:
cleaned_df['ride_duration_minutes'] = (
    cleaned_df['ended_at'] - cleaned_df['started_at']
).dt.total_seconds() / 60

cleaned_df.head()

In [ ]:
before = len(cleaned_df)

cleaned_df = cleaned_df[
    cleaned_df['ride_duration_minutes'] >= 0
].copy()

print("Invalid negative-duration records removed:",
      before - len(cleaned_df))

In [ ]:
station_columns = [
    'start_station_name',
    'end_station_name'
]

for column in station_columns:
    cleaned_df[column] = cleaned_df[column].fillna('Unknown')

print("Missing station names after handling:")
print(cleaned_df[station_columns].isnull().sum())

## 4. Outlier Detection

Outliers are unusual values that differ significantly from the general distribution of the data.

The Interquartile Range (IQR) method is used to identify unusually long ride durations.

Outliers are investigated rather than automatically removed because some long rides may represent genuine customer behavior.


In [ ]:
# Visualize ride duration

plt.figure(figsize=(10, 5))
sns.boxplot(x=cleaned_df['ride_duration_minutes'])

plt.title("Box Plot of Bike Ride Duration")
plt.xlabel("Ride Duration (Minutes)")
plt.show()

In [ ]:
Q1 = cleaned_df['ride_duration_minutes'].quantile(0.25)
Q3 = cleaned_df['ride_duration_minutes'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = cleaned_df[
    (cleaned_df['ride_duration_minutes'] < lower_limit) |
    (cleaned_df['ride_duration_minutes'] > upper_limit)
]

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Number of outliers:", len(outliers))

## 5. Date Filtering

The dataset is filtered to include rides starting between October 1 and October 31, 2025. This ensures that the demand analysis is based on the selected monthly dataset.

In [ ]:
october_df = cleaned_df[
    (cleaned_df['started_at'] >= '2025-10-01') &
    (cleaned_df['started_at'] < '2025-11-01')
].copy()

print("October dataset shape:", october_df.shape)
print("First ride:", october_df['started_at'].min())
print("Last ride:", october_df['started_at'].max())

## 6. Exploratory Data Analysis

Exploratory Data Analysis is performed using statistical summaries and visualizations.

The analysis includes:

- Distribution of bike types.
- Comparison of member and casual riders.
- Ride duration distribution.
- Rental activity over time.

These visualizations help identify customer behavior and patterns in bike usage.

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=october_df,
    x='rideable_type'
)

plt.title("Distribution of Bike Types")
plt.xlabel("Bike Type")
plt.ylabel("Number of Rides")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=october_df,
    x='member_casual'
)

plt.title("Member vs Casual Riders")
plt.xlabel("Customer Type")
plt.ylabel("Number of Rides")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=october_df,
    x='ride_duration_minutes',
    bins=50
)

plt.xlim(0, 120)
plt.title("Distribution of Ride Duration")
plt.xlabel("Duration (Minutes)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=october_df,
    x='rideable_type',
    hue='member_casual'
)

plt.title("Bike Type Usage by Customer Category")
plt.xlabel("Bike Type")
plt.ylabel("Number of Rides")
plt.show()

## 7. Hourly Rental Demand Aggregation

The original trip-level dataset is converted into an hourly demand dataset.

Each ride is grouped according to its starting hour. The number of rides in each hour is calculated as the rental demand.

The resulting target variable is `rental_count`, which represents the number of bike rentals during a particular hour.


In [ ]:
# Group rides by their starting hour

hourly_demand = (
    october_df
    .set_index('started_at')
    .resample('h')
    .size()
    .rename('rental_count')
    .reset_index()
)

hourly_demand.rename(
    columns={'started_at': 'hour_timestamp'},
    inplace=True
)

print("Hourly dataset shape:", hourly_demand.shape)

hourly_demand.head(10)

In [ ]:
# Create a complete hourly timeline for October

full_hours = pd.date_range(
    start='2025-10-01 00:00:00',
    end='2025-10-31 23:00:00',
    freq='h'
)

hourly_demand = (
    hourly_demand
    .set_index('hour_timestamp')
    .reindex(full_hours, fill_value=0)
    .rename_axis('hour_timestamp')
    .reset_index()
)

print("Total hourly records:", len(hourly_demand))
print(hourly_demand.head())

## 8. Hourly Demand Visualization

The hourly rental demand is visualized using a line chart to understand changes in bike rental activity throughout October 2025.

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    hourly_demand['hour_timestamp'],
    hourly_demand['rental_count']
)

plt.title("Hourly Bike Rental Demand - October 2025")
plt.xlabel("Date")
plt.ylabel("Number of Rentals")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Feature Engineering

Feature engineering is used to extract useful information from the timestamp column.

The following features are created:

| Feature | Description | Expected benefit |
|---|---|---|
| hour | Hour of the day | Captures daily demand patterns |
| day_of_week | Day number | Captures weekday and weekend patterns |
| month | Month of the year | Captures seasonal variation |
| is_weekend | Weekend indicator | Identifies weekend behavior |
| is_rush_hour | Rush-hour indicator | Captures commuting demand |
| hour_sin and hour_cos | Cyclic hour features | Represents the repeating nature of time |

These features help machine learning models understand temporal patterns in bike rental demand.

In [ ]:
# Extract date and time features

hourly_demand['hour'] = (
    hourly_demand['hour_timestamp'].dt.hour
)

hourly_demand['day_of_week'] = (
    hourly_demand['hour_timestamp'].dt.dayofweek
)

hourly_demand['month'] = (
    hourly_demand['hour_timestamp'].dt.month
)

hourly_demand['day'] = (
    hourly_demand['hour_timestamp'].dt.day
)

# Weekend: Saturday = 5, Sunday = 6

hourly_demand['is_weekend'] = (
    hourly_demand['day_of_week'] >= 5
).astype(int)

# Rush hour: morning 7-9 and evening 16-19

hourly_demand['is_rush_hour'] = (
    hourly_demand['hour'].isin([7, 8, 9, 16, 17, 18, 19])
).astype(int)

# Cyclic encoding for hour

hourly_demand['hour_sin'] = np.sin(
    2 * np.pi * hourly_demand['hour'] / 24
)

hourly_demand['hour_cos'] = np.cos(
    2 * np.pi * hourly_demand['hour'] / 24
)

hourly_demand.head()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=hourly_demand,
    x='hour',
    y='rental_count',
    errorbar=None
)

plt.title("Average Bike Rental Demand by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Average Rental Count")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=hourly_demand,
    x='day_of_week',
    y='rental_count',
    errorbar=None
)

plt.title("Average Bike Rental Demand by Day of Week")
plt.xlabel("Day (0 = Monday, 6 = Sunday)")
plt.ylabel("Average Rental Count")
plt.show()

## 10. Categorical Encoding

Categorical variables are converted into numerical representations using One-Hot Encoding.

The `rideable_type` and `member_casual` columns are selected as examples.

One-Hot Encoding creates separate binary columns for each category, allowing machine learning algorithms to process categorical information without assuming an artificial numerical order.

In [ ]:
# Select categorical columns

categorical_data = october_df[
    ['rideable_type', 'member_casual']
].head(10)

# Apply One-Hot Encoding

encoded_data = pd.get_dummies(
    categorical_data,
    columns=['rideable_type', 'member_casual'],
    dtype=int
)

print("Original categorical data:")
display(categorical_data)

print("Encoded data:")
display(encoded_data)

## 11. Numerical Feature Scaling

StandardScaler is used to standardize numerical features.

It transforms the features using their mean and standard deviation.

The dataset is divided chronologically into training and testing portions. The scaler is fitted only on training data to prevent information from the test set leaking into the training process.

The target variable `rental_count` is not scaled in this preprocessing demonstration.

In [ ]:
# Select features for the demonstration

feature_columns = [
    'hour',
    'day_of_week',
    'month',
    'day',
    'is_weekend',
    'is_rush_hour',
    'hour_sin',
    'hour_cos'
]

X = hourly_demand[feature_columns]
y = hourly_demand['rental_count']

# Chronological 80:20 split

split_index = int(len(hourly_demand) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Scale numerical features

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data:", X_train_scaled.shape)
print("Testing data:", X_test_scaled.shape)
print("Scaling completed successfully!")

## 12. Final Data Validation

The processed hourly dataset is checked to ensure that:

- No duplicate hourly timestamps exist.
- Rental counts are non-negative.
- Required engineered features are available.
- Missing values are handled.
- The dataset contains the expected number of hourly records.

In [ ]:
print("Total hourly records:", len(hourly_demand))

print(
    "Duplicate timestamps:",
    hourly_demand['hour_timestamp'].duplicated().sum()
)

print(
    "Missing values:",
    hourly_demand.isnull().sum().sum()
)

print(
    "Negative rental counts:",
    (hourly_demand['rental_count'] < 0).sum()
)

print("\nFinal dataset columns:")
print(hourly_demand.columns.tolist())

print("\nFinal dataset preview:")
display(hourly_demand.head())

## 13. Saving the Processed Dataset

The final hourly dataset is exported as a CSV file. This file will be used in the upcoming machine learning model development phase.

The processed dataset contains the hourly rental demand target and engineered time-based features.

In [ ]:
# Save processed hourly dataset

output_file = "processed_bike_rental_demand_october_2025.csv"

hourly_demand.to_csv(
    output_file,
    index=False
)

print("Processed dataset saved successfully!")

In [ ]:
from google.colab import files

files.download(output_file)

## 14. Conclusion

In Week 2, the Capital Bikeshare October 2025 trip-history dataset was successfully prepared for bike rental demand prediction.

The original trip-level data was inspected, cleaned, and analyzed. Missing station information was handled, duplicate records were checked, and unusually long ride durations were investigated using the IQR method.

The individual bike rental records were aggregated into hourly rental demand. New time-based features, including hour, day of the week, weekend indicator, rush-hour indicator, and cyclic time features, were created.

Exploratory visualizations were generated to understand bike usage patterns. Categorical encoding and numerical scaling were also demonstrated.

The final processed dataset was exported as a CSV file for use in the upcoming machine learning model development phase.

This completes the main data preprocessing and feature engineering activities for Week 2.